In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, itertools, pathlib
warnings.filterwarnings('ignore')
pathlib.Path('figs').mkdir(exist_ok=True)
_TAG = 'nb2'
_fig_counter = itertools.count(1)
def _save_show(*a, **k):
    import matplotlib.pyplot as _plt
    for _n in _plt.get_fignums():
        _plt.figure(_n).savefig('figs/%s_fig%02d.png' % (_TAG, next(_fig_counter)), dpi=140, bbox_inches='tight')
    _plt.close('all')
plt.show = _save_show


# Characterizing Orbital Perturbations via Physics-Informed Neural Networks  
## Notebook 2: Real Dataset Import, Unified Preprocessing, and Benchmark-Ready Observation Tables

This notebook is the follow-up to **Notebook 1**, where the project created a controlled synthetic dataset with a known hidden perturbation parameter.

Notebook 1 answered the question:

> **Can we generate a clean inverse-problem dataset where the true physics is known?**

Notebook 2 answers the next question:

> **How do we begin bringing real orbital data into the project in a structured, reusable way?**


### Goals for this notebook
1. Reload the synthetic data products from Notebook 1
2. Define one **unified observation format** for all datasets
3. Import a public **GP / TLE-style dataset** using CelesTrak
4. Import a **high-precision benchmark dataset** using NASA JPL Horizons
5. Save all imported data into project-ready tables for later PINN work!

A strong PINN project does not jump directly from a toy simulation to a neural network.  
Before training any model, the data pipeline should be clear:

- where the data comes from
- what the units are
- how timestamps are represented
- what the state variables mean
- how different sources can be converted into a common format



## 1. Where this notebook fits in the project

The current project workflow is:

1. **Synthetic trajectories with known hidden parameter**  
   Used for controlled validation and honest testing of inverse recovery

2. **Real orbital tracking data (GP / TLE family)**  
   Used to test whether the method still works when the data is sparse, practical, and messier

3. **High-precision benchmark ephemerides**  
   Used to compare against a much more precise reference source

This notebook begins steps 2 and 3 without abandoning step 1.  
That is important because the synthetic dataset remains the scientific control case for the project.

For live public access, this notebook starts with:
- **CelesTrak** for public GP / TLE-style access
- **NASA JPL Horizons** for high-precision state vectors

Later, once a Space-Track account is active, the same preprocessing logic can be reused there too.


## 2. Environment setup

This notebook uses:
- `numpy`
- `pandas`
- `matplotlib`
- `requests`
- `pathlib`

In [2]:
import csv
import io
import json
from pathlib import Path
from urllib.parse import urlencode

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=6, suppress=True)

DATA_DIR = Path(".")

print("Working directory:", DATA_DIR.resolve())

Working directory: /private/tmp/pinn_run/main


In [3]:
# [neutralized for headless run]
# %pip install sgp4 #for TLE data

## 3. Reload the synthetic outputs from Notebook 1

Notebook 1 should have saved three files:

- `synthetic_dense_truth.csv`
- `synthetic_sparse_observations.csv`
- `synthetic_metadata.csv`

The first job here is simply to reload them and verify that the synthetic branch of the project is still intact.


In [4]:
dense_path = DATA_DIR / "synthetic_dense_truth.csv"
sparse_path = DATA_DIR / "synthetic_sparse_observations.csv"
metadata_path = DATA_DIR / "synthetic_metadata.csv"

synthetic_dense = pd.read_csv(dense_path) if dense_path.exists() else None
synthetic_sparse = pd.read_csv(sparse_path) if sparse_path.exists() else None
synthetic_meta = pd.read_csv(metadata_path) if metadata_path.exists() else None

print("synthetic_dense found:", dense_path.exists())
print("synthetic_sparse found:", sparse_path.exists())
print("synthetic_metadata found:", metadata_path.exists())

if synthetic_dense is not None:
    print("\nDense synthetic dataset preview:")
    display(synthetic_dense.head())

if synthetic_sparse is not None:
    print("\nSparse synthetic dataset preview:")
    display(synthetic_sparse.head())

if synthetic_meta is not None:
    print("\nSynthetic metadata:")
    display(synthetic_meta)

synthetic_dense found: True
synthetic_sparse found: True
synthetic_metadata found: True

Dense synthetic dataset preview:


,t_sec,x_km,y_km,z_km,vx_kms,vy_kms,vz_kms
0,0.000000,6878.136300,0.000000,0.000000,0.000000,6.235884,4.366413
1,18.173596,6876.743017,113.320723,79.348009,-0.153325,6.234614,4.365521
2,36.347193,6872.563737,226.595413,158.663697,-0.306588,6.230818,4.362856
3,54.520789,6865.600154,339.778180,237.914840,-0.459727,6.224498,4.358418
4,72.694385,6855.855095,452.823170,317.069241,-0.612679,6.215656,4.352210



Sparse synthetic dataset preview:


,t_sec,x_km,y_km,z_km,vx_kms,vy_kms,vz_kms
0,0.000000,6878.288659,-0.519992,0.375226,0.001076,6.238559,4.366104
1,636.075872,5241.243801,3645.400783,2551.959344,-4.934978,4.750742,3.324524
2,1272.151745,1108.203840,5556.111180,3886.349185,-7.518461,1.000688,0.693901
3,1908.227617,-3552.201508,4817.853566,3362.843518,-6.519707,-3.225914,-2.269638
4,2544.303490,-6512.758969,1777.165964,1227.755373,-2.399246,-5.931114,-4.160952



Synthetic metadata:


,parameter,value
0,true_drag_coeff,3.245e-05
1,use_j2,True
2,position_noise_std_km,0.5
3,velocity_noise_std_kms,0.002
4,observe_every,35
5,altitude_km,500.0
6,inclination_deg,35.0
7,n_dense_points,2500
8,n_sparse_points,72


## 4. Define a unified observation schema

Different orbital datasets rarely arrive in the same structure.

For example:
- the synthetic dataset from Notebook 1 uses `t_sec`
- TLE data arrives as mean-element information around an epoch
- Horizons can return state vectors with calendar times or Julian dates

To keep the later PINN work organized, we will define one standard table format.

### Standard columns we want
- `dataset_name`
- `object_name`
- `source_name`
- `timestamp_utc`
- `t_sec`
- `x_km`, `y_km`, `z_km`
- `vx_kms`, `vy_kms`, `vz_kms`

Once data from any source can be converted into that format, the rest of the project becomes much easier.


In [5]:
EARTH_RADIUS_KM = 6378.1363

def ensure_utc_datetime(series):
    return pd.to_datetime(series, utc=True)

def add_time_from_start_seconds(df, time_col="timestamp_utc"):
    df = df.copy()
    df[time_col] = ensure_utc_datetime(df[time_col])
    df["t_sec"] = (df[time_col] - df[time_col].iloc[0]).dt.total_seconds()
    return df

def build_standard_frame(
    raw_df,
    timestamp_col,
    position_cols,
    velocity_cols,
    dataset_name,
    object_name,
    source_name,
):
    out = pd.DataFrame({
        "dataset_name": dataset_name,
        "object_name": object_name,
        "source_name": source_name,
        "timestamp_utc": ensure_utc_datetime(raw_df[timestamp_col]),
        "x_km": pd.to_numeric(raw_df[position_cols[0]], errors="coerce"),
        "y_km": pd.to_numeric(raw_df[position_cols[1]], errors="coerce"),
        "z_km": pd.to_numeric(raw_df[position_cols[2]], errors="coerce"),
        "vx_kms": pd.to_numeric(raw_df[velocity_cols[0]], errors="coerce"),
        "vy_kms": pd.to_numeric(raw_df[velocity_cols[1]], errors="coerce"),
        "vz_kms": pd.to_numeric(raw_df[velocity_cols[2]], errors="coerce"),
    })
    out = add_time_from_start_seconds(out, "timestamp_utc")
    return out

def add_radius_speed_altitude(df):
    df = df.copy()
    r = np.sqrt(df["x_km"]**2 + df["y_km"]**2 + df["z_km"]**2)
    v = np.sqrt(df["vx_kms"]**2 + df["vy_kms"]**2 + df["vz_kms"]**2)
    df["radius_km"] = r
    df["speed_kms"] = v
    df["altitude_km"] = r - EARTH_RADIUS_KM
    return df

def summarize_dataset(df):
    if df is None or len(df) == 0:
        return None
    dt = df["timestamp_utc"].sort_values().diff().dt.total_seconds().dropna()
    return pd.DataFrame({
        "n_points": [len(df)],
        "start_time_utc": [df["timestamp_utc"].min()],
        "end_time_utc": [df["timestamp_utc"].max()],
        "median_step_sec": [dt.median() if len(dt) else np.nan],
        "min_radius_km": [df["radius_km"].min() if "radius_km" in df.columns else np.nan],
        "max_radius_km": [df["radius_km"].max() if "radius_km" in df.columns else np.nan],
    })

## 5. Convert the synthetic trajectory into the same standard format

The synthetic data does not come with absolute UTC timestamps.  
That is fine for model development, but for a unified project pipeline we want to place it on a notional timeline.

This does **not** change the physics. It just gives the table a consistent timestamp column so that later plotting and comparison code can be reused.


In [6]:
synthetic_standard = None

if synthetic_dense is not None:
    synthetic_epoch = pd.Timestamp("2026-01-01T00:00:00Z")
    synthetic_dense_with_time = synthetic_dense.copy()
    synthetic_dense_with_time["timestamp_utc"] = synthetic_epoch + pd.to_timedelta(
        synthetic_dense_with_time["t_sec"], unit="s"
    )

    synthetic_standard = build_standard_frame(
        synthetic_dense_with_time,
        timestamp_col="timestamp_utc",
        position_cols=["x_km", "y_km", "z_km"],
        velocity_cols=["vx_kms", "vy_kms", "vz_kms"],
        dataset_name="synthetic_dense_truth",
        object_name="synthetic_orbit_case",
        source_name="notebook_1_simulation",
    )
    synthetic_standard = add_radius_speed_altitude(synthetic_standard)

    display(synthetic_standard.head())
else:
    print("Synthetic files are not yet available. Run Notebook 1 first, then return here.")

,dataset_name,object_name,source_name,timestamp_utc,x_km,y_km,z_km,vx_kms,vy_kms,vz_kms,t_sec,radius_km,speed_kms,altitude_km
0,synthetic_dense_truth,synthetic_orbit_case,notebook_1_simulation,2026-01-01 00:00:00+00:00,6878.136300,0.000000,0.000000,0.000000,6.235884,4.366413,0.000000,6878.136300,7.612609,500.000000
1,synthetic_dense_truth,synthetic_orbit_case,notebook_1_simulation,2026-01-01 00:00:18.173596357+00:00,6876.743017,113.320723,79.348009,-0.153325,6.234614,4.365521,18.173596,6878.134356,7.612601,499.998056
2,synthetic_dense_truth,synthetic_orbit_case,notebook_1_simulation,2026-01-01 00:00:36.347192713+00:00,6872.563737,226.595413,158.663697,-0.306588,6.230818,4.362856,36.347193,6878.128522,7.612595,499.992222
3,synthetic_dense_truth,synthetic_orbit_case,notebook_1_simulation,2026-01-01 00:00:54.520789069+00:00,6865.600154,339.778180,237.914840,-0.459727,6.224498,4.358418,54.520789,6878.118795,7.612591,499.982495
4,synthetic_dense_truth,synthetic_orbit_case,notebook_1_simulation,2026-01-01 00:01:12.694385426+00:00,6855.855095,452.823170,317.069241,-0.612679,6.215656,4.352210,72.694385,6878.105176,7.612588,499.968876


In [7]:
if synthetic_standard is not None:
    fig, ax = plt.subplots()
    ax.plot(synthetic_standard["x_km"], synthetic_standard["y_km"])
    ax.set_xlabel("x (km)")
    ax.set_ylabel("y (km)")
    ax.set_title("Synthetic dataset reloaded in standard format")
    ax.axis("equal")
    plt.show()

    display(summarize_dataset(synthetic_standard))

,n_points,start_time_utc,end_time_utc,median_step_sec,min_radius_km,max_radius_km
0,2500,2026-01-01 00:00:00+00:00,2026-01-01 12:36:55.817295136+00:00,18.173596,6800.227964,6878.1363


## 6. Import a public GP / TLE-style dataset with CelesTrak

The project plan discussed Space-Track TLEs.  
For development, a very practical public starting point is **CelesTrak**, which exposes GP data through public query URLs.

This section uses two related CelesTrak products for the **same satellite**:

1. **OMM-style JSON** for easy metadata inspection
2. **TLE text** for optional propagation with SGP4

We will use the ISS as the first worked example because it is a familiar, stable target and is commonly used in demonstrations.

### Useful query pattern
CelesTrak GP queries follow this structure:

`https://celestrak.org/NORAD/elements/gp.php?CATNR=25544&FORMAT=...`

We will first request:
- JSON for inspection
- TLE text for propagation


In [8]:
CELESTRAK_CATNR = 25544
CELESTRAK_OBJECT_NAME = "ISS (ZARYA)"

celestrak_json_url = f"https://celestrak.org/NORAD/elements/gp.php?CATNR={CELESTRAK_CATNR}&FORMAT=JSON"
celestrak_tle_url = f"https://celestrak.org/NORAD/elements/gp.php?CATNR={CELESTRAK_CATNR}&FORMAT=TLE"

print("CelesTrak JSON URL:", celestrak_json_url)
print("CelesTrak TLE URL :", celestrak_tle_url)

celestrak_json_data = None
celestrak_tle_text = None

try:
    celestrak_json_resp = requests.get(celestrak_json_url, timeout=30)
    celestrak_json_resp.raise_for_status()
    celestrak_json_data = celestrak_json_resp.json()
    print("Successfully downloaded CelesTrak JSON.")
except Exception as e:
    print("Could not download CelesTrak JSON:", e)

try:
    celestrak_tle_resp = requests.get(celestrak_tle_url, timeout=30)
    celestrak_tle_resp.raise_for_status()
    celestrak_tle_text = celestrak_tle_resp.text
    print("Successfully downloaded CelesTrak TLE text.")
except Exception as e:
    print("Could not download CelesTrak TLE text:", e)

CelesTrak JSON URL: https://celestrak.org/NORAD/elements/gp.php?CATNR=25544&FORMAT=JSON
CelesTrak TLE URL : https://celestrak.org/NORAD/elements/gp.php?CATNR=25544&FORMAT=TLE


Successfully downloaded CelesTrak JSON.


Successfully downloaded CelesTrak TLE text.


## 7. Inspect the CelesTrak OMM-style metadata

Even before propagation, the GP / OMM record itself is useful.

It contains values such as:
- the epoch
- inclination
- eccentricity
- mean motion
- BSTAR
- the NORAD catalog number
- the object name

This is important because the project should not treat a real orbital dataset like a black box.  
You should understand what the data source is actually providing.


In [9]:
celestrak_omm_df = None

if celestrak_json_data is not None:
    celestrak_omm_df = pd.DataFrame(celestrak_json_data)
    display(celestrak_omm_df.head())

    numeric_candidates = [
        "NORAD_CAT_ID", "MEAN_MOTION", "ECCENTRICITY", "INCLINATION",
        "RA_OF_ASC_NODE", "ARG_OF_PERICENTER", "MEAN_ANOMALY",
        "BSTAR", "EPHEMERIS_TYPE", "ELEMENT_SET_NO", "REV_AT_EPOCH"
    ]
    for col in numeric_candidates:
        if col in celestrak_omm_df.columns:
            celestrak_omm_df[col] = pd.to_numeric(celestrak_omm_df[col], errors="coerce")

    if "EPOCH" in celestrak_omm_df.columns:
        celestrak_omm_df["EPOCH"] = pd.to_datetime(celestrak_omm_df["EPOCH"], utc=True)

    print("\nAvailable columns:")
    print(list(celestrak_omm_df.columns))
else:
    print("No CelesTrak JSON record available yet.")

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT
0,ISS (ZARYA),1998-067A,2026-06-18T19:14:58.749504,15.493025,0.000467,51.6331,292.1003,202.292,157.7867,0,U,25544,999,57200,0.000154,0.000082,0



Available columns:
['OBJECT_NAME', 'OBJECT_ID', 'EPOCH', 'MEAN_MOTION', 'ECCENTRICITY', 'INCLINATION', 'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER', 'MEAN_ANOMALY', 'EPHEMERIS_TYPE', 'CLASSIFICATION_TYPE', 'NORAD_CAT_ID', 'ELEMENT_SET_NO', 'REV_AT_EPOCH', 'BSTAR', 'MEAN_MOTION_DOT', 'MEAN_MOTION_DDOT']


In [10]:
if celestrak_omm_df is not None and len(celestrak_omm_df) > 0:
    omm_row = celestrak_omm_df.iloc[0]
    interesting_cols = [
        col for col in [
            "OBJECT_NAME", "OBJECT_ID", "NORAD_CAT_ID", "EPOCH", "INCLINATION",
            "ECCENTRICITY", "MEAN_MOTION", "BSTAR", "MEAN_ANOMALY",
            "RA_OF_ASC_NODE", "ARG_OF_PERICENTER"
        ] if col in celestrak_omm_df.columns
    ]
    display(omm_row[interesting_cols].to_frame(name="value"))

,value
OBJECT_NAME,ISS (ZARYA)
OBJECT_ID,1998-067A
NORAD_CAT_ID,25544
EPOCH,2026-06-18 19:14:58.749504+00:00
INCLINATION,51.6331
ECCENTRICITY,0.000467
MEAN_MOTION,15.493025
BSTAR,0.000154
MEAN_ANOMALY,157.7867
RA_OF_ASC_NODE,292.1003


## 8. Optional TLE propagation with SGP4

A TLE is not a dense time series by itself.  
It is a compact mean-element description centered around an epoch.

To turn that into position and velocity samples, we need a propagator.  
For standard TLE work, the appropriate starting point is **SGP4**.

This section:
1. parses the downloaded TLE text
2. converts it into an SGP4 satellite object
3. propagates it over a short window around the TLE epoch
4. builds a state-vector time series in kilometers and kilometers per second

### Important conceptual point
This is the first real place where the project begins to separate:
- **data source** from
- **propagation model**

That distinction matters later for the inverse problem.


In [11]:
try:
    from sgp4.api import Satrec, jday
    SGP4_AVAILABLE = True
except Exception as e:
    SGP4_AVAILABLE = False
    print("sgp4 is not available in this environment:", e)

def parse_tle_block(tle_text):
    lines = [line.rstrip() for line in tle_text.splitlines() if line.strip()]
    if len(lines) >= 3 and lines[1].startswith("1 ") and lines[2].startswith("2 "):
        return lines[0], lines[1], lines[2]
    if len(lines) >= 2 and lines[0].startswith("1 ") and lines[1].startswith("2 "):
        return "UNKNOWN_OBJECT", lines[0], lines[1]
    raise ValueError("Could not parse TLE block.")

def jd_to_timestamp_utc(jd):
    unix_seconds = (jd - 2440587.5) * 86400.0
    return pd.to_datetime(unix_seconds, unit="s", utc=True)

def timestamp_to_jday(ts):
    ts = pd.Timestamp(ts).tz_convert("UTC")
    sec = ts.second + ts.microsecond / 1e6
    return jday(ts.year, ts.month, ts.day, ts.hour, ts.minute, sec)

def propagate_tle_to_dataframe(satrec, timestamps_utc, object_name, source_name):
    records = []
    for ts in timestamps_utc:
        jd, fr = timestamp_to_jday(ts)
        error_code, r_km, v_kms = satrec.sgp4(jd, fr)
        records.append({
            "timestamp_utc": pd.Timestamp(ts).tz_convert("UTC"),
            "error_code": error_code,
            "x_km": r_km[0] if error_code == 0 else np.nan,
            "y_km": r_km[1] if error_code == 0 else np.nan,
            "z_km": r_km[2] if error_code == 0 else np.nan,
            "vx_kms": v_kms[0] if error_code == 0 else np.nan,
            "vy_kms": v_kms[1] if error_code == 0 else np.nan,
            "vz_kms": v_kms[2] if error_code == 0 else np.nan,
        })

    df = pd.DataFrame(records)
    df["dataset_name"] = "celestrak_tle_propagated"
    df["object_name"] = object_name
    df["source_name"] = source_name
    df = add_time_from_start_seconds(df, "timestamp_utc")
    df = add_radius_speed_altitude(df)
    return df

In [12]:
celestrak_standard = None

if SGP4_AVAILABLE and celestrak_tle_text:
    tle_name, tle_line1, tle_line2 = parse_tle_block(celestrak_tle_text)
    sat = Satrec.twoline2rv(tle_line1, tle_line2)

    tle_epoch_jd = sat.jdsatepoch + sat.jdsatepochF
    tle_epoch_ts = jd_to_timestamp_utc(tle_epoch_jd)
    print("Parsed TLE object name:", tle_name)
    print("TLE epoch (UTC):", tle_epoch_ts)

    propagation_minutes = np.arange(0, 24 * 60 + 10, 10)
    propagation_times = tle_epoch_ts + pd.to_timedelta(propagation_minutes, unit="m")

    celestrak_standard = propagate_tle_to_dataframe(
        sat,
        propagation_times,
        object_name=tle_name,
        source_name="celestrak_tle_sgp4",
    )

    print("Number of propagated states:", len(celestrak_standard))
    display(celestrak_standard.head())
else:
    print("Skipping propagation. Either sgp4 is missing or the TLE could not be downloaded.")

Parsed TLE object name: ISS (ZARYA)
TLE epoch (UTC): 2026-06-18 19:14:58.749487162+00:00
Number of propagated states: 145


,timestamp_utc,error_code,x_km,y_km,z_km,vx_kms,vy_kms,vz_kms,dataset_name,object_name,source_name,t_sec,radius_km,speed_kms,altitude_km
0,2026-06-18 19:14:58.749487162+00:00,0,2558.580310,-6300.915911,0.006133,4.399827,1.792287,6.005514,celestrak_tle_propagated,ISS (ZARYA),celestrak_tle_sgp4,0.0,6800.578984,7.657478,422.442684
1,2026-06-18 19:24:58.749487162+00:00,0,4439.200507,-3919.803615,3334.285750,1.628275,5.840270,4.680973,celestrak_tle_propagated,ISS (ZARYA),celestrak_tle_sgp4,600.0,6796.235942,7.659736,418.099642
2,2026-06-18 19:34:58.749487162+00:00,0,4367.389777,185.452078,5198.125232,-1.858123,7.318800,1.293124,celestrak_tle_propagated,ISS (ZARYA),celestrak_tle_sgp4,1200.0,6791.832733,7.660916,413.696433
3,2026-06-18 19:44:58.749487162+00:00,0,2375.278699,4209.089099,4770.064508,-4.527977,5.578288,-2.665895,celestrak_tle_propagated,ISS (ZARYA),celestrak_tle_sgp4,1800.0,6790.574007,7.663346,412.437707
4,2026-06-18 19:54:58.749487162+00:00,0,-661.993891,6379.572365,2236.804106,-5.207301,1.377223,-5.453665,celestrak_tle_propagated,ISS (ZARYA),celestrak_tle_sgp4,2400.0,6792.677829,7.665193,414.541529


In [13]:
if celestrak_standard is not None:
    fig, ax = plt.subplots()
    ax.plot(celestrak_standard["x_km"], celestrak_standard["y_km"])
    ax.set_xlabel("x (km)")
    ax.set_ylabel("y (km)")
    ax.set_title("CelesTrak / SGP4 propagated trajectory in the xy-plane")
    ax.axis("equal")
    plt.show()

    fig, ax = plt.subplots()
    ax.plot(celestrak_standard["t_sec"] / 3600.0, celestrak_standard["altitude_km"])
    ax.set_xlabel("Time since TLE epoch (hours)")
    ax.set_ylabel("Altitude (km)")
    ax.set_title("Altitude over time from propagated TLE")
    plt.show()

    display(summarize_dataset(celestrak_standard))

,n_points,start_time_utc,end_time_utc,median_step_sec,min_radius_km,max_radius_km
0,145,2026-06-18 19:14:58.749487162+00:00,2026-06-19 19:14:58.749487162+00:00,600.0,6790.540378,6802.017391


## 9. Import a high-precision benchmark dataset from NASA JPL Horizons

Now we bring in the benchmark branch of the workflow.

Horizons can return:
- observer ephemerides
- state vectors
- orbital elements

For this project, the most useful first format is **state vectors**, because they map naturally onto the same structure as the synthetic data:
- position
- velocity
- time

### First benchmark choice
Here we use the **Moon** as a first benchmark example because:
- it is easy to query from Horizons
- it returns clean state-vector data
- it demonstrates the benchmark ingestion workflow clearly

This is not yet the final benchmark target for the project.  
It is the first fully worked example of the import pipeline.


In [14]:
HORIZONS_API = "https://ssd.jpl.nasa.gov/api/horizons.api"

def fetch_horizons_vectors(
    command="301",
    center="geo",
    start_time="2026-01-01",
    stop_time="2026-01-04",
    step_size="60 min",
    vec_table="2",
):
    params = {
        "format": "text",
        "COMMAND": f"'{command}'",
        "OBJ_DATA": "'YES'",
        "MAKE_EPHEM": "'YES'",
        "EPHEM_TYPE": "'VECTORS'",
        "CENTER": f"'{center}'",
        "START_TIME": f"'{start_time}'",
        "STOP_TIME": f"'{stop_time}'",
        "STEP_SIZE": f"'{step_size}'",
        "CSV_FORMAT": "'YES'",
        "REF_SYSTEM": "'ICRF'",
        "OUT_UNITS": "'KM-S'",
        "VEC_TABLE": f"'{vec_table}'",
        "VEC_CORR": "'NONE'",
        "TIME_TYPE": "'UT'",
    }

    response = requests.get(HORIZONS_API, params=params, timeout=60)
    response.raise_for_status()
    return response.text, response.url

def parse_horizons_vector_text(text):
    if "$$SOE" not in text or "$$EOE" not in text:
        raise ValueError("Could not find $$SOE / $$EOE markers in Horizons output.")

    body = text.split("$$SOE")[1].split("$$EOE")[0].strip()
    rows = []

    for line in body.splitlines():
        line = line.strip()
        if not line:
            continue
        parsed = next(csv.reader([line]))
        parsed = [item.strip() for item in parsed]

        # For VEC_TABLE='2' with CSV_FORMAT='YES', we expect at least:
        # JD, Calendar Date, X, Y, Z, VX, VY, VZ
        if len(parsed) < 8:
            continue

        rows.append({
            "jd_tdb_or_ut": parsed[0],
            "calendar_date": parsed[1],
            "x_km": parsed[2],
            "y_km": parsed[3],
            "z_km": parsed[4],
            "vx_kms": parsed[5],
            "vy_kms": parsed[6],
            "vz_kms": parsed[7],
        })

    if not rows:
        raise ValueError("No vector rows were parsed from Horizons output.")

    df = pd.DataFrame(rows)
    for col in ["x_km", "y_km", "z_km", "vx_kms", "vy_kms", "vz_kms", "jd_tdb_or_ut"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # The calendar field usually looks like 'A.D. 2026-Jan-01 00:00:00.0000 UT'
    df["timestamp_utc"] = (
        df["calendar_date"]
        .str.replace("A.D. ", "", regex=False)
        .str.replace(" UT", "", regex=False)
    )
    df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)

    return df

In [15]:
horizons_standard = None
horizons_raw_text = None
horizons_request_url = None

try:
    horizons_raw_text, horizons_request_url = fetch_horizons_vectors(
        command="301",      # Moon
        center="geo",       # Geocenter
        start_time="2026-01-01",
        stop_time="2026-01-04",
        step_size="60 min",
        vec_table="2",
    )
    print("Horizons request URL:")
    print(horizons_request_url)

    horizons_df = parse_horizons_vector_text(horizons_raw_text)

    horizons_standard = build_standard_frame(
        horizons_df,
        timestamp_col="timestamp_utc",
        position_cols=["x_km", "y_km", "z_km"],
        velocity_cols=["vx_kms", "vy_kms", "vz_kms"],
        dataset_name="horizons_vectors",
        object_name="Moon",
        source_name="nasa_jpl_horizons",
    )
    horizons_standard = add_radius_speed_altitude(horizons_standard)

    display(horizons_standard.head())
except Exception as e:
    print("Could not download or parse Horizons data:", e)

Horizons request URL:
https://ssd.jpl.nasa.gov/api/horizons.api?format=text&COMMAND=%27301%27&OBJ_DATA=%27YES%27&MAKE_EPHEM=%27YES%27&EPHEM_TYPE=%27VECTORS%27&CENTER=%27geo%27&START_TIME=%272026-01-01%27&STOP_TIME=%272026-01-04%27&STEP_SIZE=%2760+min%27&CSV_FORMAT=%27YES%27&REF_SYSTEM=%27ICRF%27&OUT_UNITS=%27KM-S%27&VEC_TABLE=%272%27&VEC_CORR=%27NONE%27&TIME_TYPE=%27UT%27


,dataset_name,object_name,source_name,timestamp_utc,x_km,y_km,z_km,vx_kms,vy_kms,vz_kms,t_sec,radius_km,speed_kms,altitude_km
0,horizons_vectors,Moon,nasa_jpl_horizons,2026-01-01 00:00:00+00:00,144256.242111,329424.941296,31753.362095,-1.004401,0.420671,0.005566,0.0,361024.834797,1.088952,354646.698497
1,horizons_vectors,Moon,nasa_jpl_horizons,2026-01-01 01:00:00+00:00,140632.368555,330921.171662,31771.627096,-1.008844,0.410559,0.004581,3600.0,360964.986153,1.089195,354586.849853
2,horizons_vectors,Moon,nasa_jpl_horizons,2026-01-01 02:00:00+00:00,136992.696683,332380.908087,31786.342414,-1.013177,0.400397,0.003594,7200.0,360907.797878,1.089431,354529.661578
3,horizons_vectors,Moon,nasa_jpl_horizons,2026-01-01 03:00:00+00:00,133337.624982,333803.972919,31797.504725,-1.017399,0.390186,0.002607,10800.0,360853.288580,1.089658,354475.152280
4,horizons_vectors,Moon,nasa_jpl_horizons,2026-01-01 04:00:00+00:00,129667.554374,335190.193227,31805.111179,-1.021510,0.379928,0.001619,14400.0,360801.476425,1.089876,354423.340125


In [16]:
if horizons_standard is not None:
    fig, ax = plt.subplots()
    ax.plot(horizons_standard["x_km"], horizons_standard["y_km"])
    ax.set_xlabel("x (km)")
    ax.set_ylabel("y (km)")
    ax.set_title("NASA JPL Horizons benchmark trajectory in the xy-plane")
    ax.axis("equal")
    plt.show()

    fig, ax = plt.subplots()
    ax.plot(horizons_standard["t_sec"] / 3600.0, horizons_standard["radius_km"])
    ax.set_xlabel("Time since first benchmark sample (hours)")
    ax.set_ylabel("Radius from Earth center (km)")
    ax.set_title("Radius evolution in the Horizons benchmark dataset")
    plt.show()

    display(summarize_dataset(horizons_standard))

,n_points,start_time_utc,end_time_utc,median_step_sec,min_radius_km,max_radius_km
0,73,2026-01-01 00:00:00+00:00,2026-01-04 00:00:00+00:00,3600.0,360347.92394,364083.390861


## 10. Compare the three branches in one project language

At this point we may have up to three project-ready datasets available:

- `synthetic_standard`
- `celestrak_standard`
- `horizons_standard`

Even though these sources come from very different places, the project can now compare them in one common format.

That is exactly the infrastructure needed before:
- inverse parameter learning
- physics residual evaluation
- PINN training on sparse observation subsets


In [17]:
available_frames = {
    "synthetic_standard": synthetic_standard,
    "celestrak_standard": celestrak_standard,
    "horizons_standard": horizons_standard,
}

summary_tables = []
for name, df in available_frames.items():
    if df is not None and len(df) > 0:
        summary = summarize_dataset(df)
        summary.insert(0, "frame_name", name)
        summary_tables.append(summary)

if summary_tables:
    combined_summary = pd.concat(summary_tables, ignore_index=True)
    display(combined_summary)
else:
    print("No datasets are available yet for comparison.")

,frame_name,n_points,start_time_utc,end_time_utc,median_step_sec,min_radius_km,max_radius_km
0,synthetic_standard,2500,2026-01-01 00:00:00+00:00,2026-01-01 12:36:55.817295136+00:00,18.173596,6800.227964,6878.136300
1,celestrak_standard,145,2026-06-18 19:14:58.749487162+00:00,2026-06-19 19:14:58.749487162+00:00,600.000000,6790.540378,6802.017391
2,horizons_standard,73,2026-01-01 00:00:00+00:00,2026-01-04 00:00:00+00:00,3600.000000,360347.923940,364083.390861


In [18]:
if synthetic_standard is not None and celestrak_standard is not None:
    fig, ax = plt.subplots()
    ax.plot(synthetic_standard["x_km"], synthetic_standard["y_km"], label="Synthetic")
    ax.plot(celestrak_standard["x_km"], celestrak_standard["y_km"], label="CelesTrak propagated")
    ax.set_xlabel("x (km)")
    ax.set_ylabel("y (km)")
    ax.set_title("Synthetic orbit vs propagated real-data example")
    ax.axis("equal")
    ax.legend()
    plt.show()

if celestrak_standard is not None and horizons_standard is not None:
    fig, ax = plt.subplots()
    ax.plot(celestrak_standard["t_sec"] / 3600.0, celestrak_standard["radius_km"], label="CelesTrak propagated")
    ax.plot(horizons_standard["t_sec"] / 3600.0, horizons_standard["radius_km"], label="Horizons benchmark")
    ax.set_xlabel("Time from dataset start (hours)")
    ax.set_ylabel("Radius from Earth center (km)")
    ax.set_title("Radial scale comparison across real-data branches")
    ax.legend()
    plt.show()

## 11. Save benchmark-ready tables

Saving standardized tables now will make later notebooks much easier.

These exports are especially useful for:
- a future PINN notebook
- baseline optimizer notebooks
- exploratory plotting notebooks
- report figures and appendix tables


In [19]:
output_paths = []

if synthetic_standard is not None:
    path = DATA_DIR / "notebook2_synthetic_standardized.csv"
    synthetic_standard.to_csv(path, index=False)
    output_paths.append(path)

if celestrak_standard is not None:
    path = DATA_DIR / "notebook2_celestrak_iss_standardized.csv"
    celestrak_standard.to_csv(path, index=False)
    output_paths.append(path)

if horizons_standard is not None:
    path = DATA_DIR / "notebook2_horizons_moon_standardized.csv"
    horizons_standard.to_csv(path, index=False)
    output_paths.append(path)

for path in output_paths:
    print("Saved:", path)

Saved: notebook2_synthetic_standardized.csv
Saved: notebook2_celestrak_iss_standardized.csv
Saved: notebook2_horizons_moon_standardized.csv


## 12. Optional template for future Space-Track ingestion

The original project plan includes Space-Track as the main real TLE source.  
That remains a good target, but it requires a registered user account and authenticated access.

This template is included so you can see how the later authenticated workflow will look. It is intentionally written to use **environment variables** rather than hardcoded credentials.

Do not hardcode usernames or passwords into the notebook!


In [20]:
# Optional Space-Track template
# This cell is written as a template and may not run until credentials are available.

import os

ST_USER = os.getenv("SPACETRACK_USERNAME")
ST_PASS = os.getenv("SPACETRACK_PASSWORD")

if not ST_USER or not ST_PASS:
    print("Space-Track credentials not found in environment variables.")
    print("When available, set SPACETRACK_USERNAME and SPACETRACK_PASSWORD.")
else:
    login_url = "https://www.space-track.org/ajaxauth/login"
    query_url = (
        "https://www.space-track.org/basicspacedata/query/"
        "class/gp_history/"
        "NORAD_CAT_ID/25544/"
        "orderby/EPOCH asc/"
        "format/json"
    )

    with requests.Session() as session:
        login_resp = session.post(
            login_url,
            data={"identity": ST_USER, "password": ST_PASS},
            timeout=30,
        )
        login_resp.raise_for_status()

        data_resp = session.get(query_url, timeout=60)
        data_resp.raise_for_status()

        space_track_df = pd.DataFrame(data_resp.json())
        print("Downloaded rows from Space-Track:", len(space_track_df))
        display(space_track_df.head())

Space-Track credentials not found in environment variables.
When available, set SPACETRACK_USERNAME and SPACETRACK_PASSWORD.


## 13. What this notebook accomplished

This notebook extends the project in an important way.

### New capabilities added here
- the synthetic dataset from Notebook 1 can be reloaded cleanly
- the project now has a **standard observation schema**
- a public **GP / TLE-style data source** can be ingested
- a **high-precision benchmark source** can be ingested
- multiple sources can now be saved in a common table format

### Why that matters scientifically
A PINN will only be persuasive if the data pathway is disciplined.  
This notebook establishes that discipline.

The project is no longer just:
- one simulation
- one figure
- one hidden parameter

It now has the beginning of a true multi-source orbital data pipeline.


## 14. Checklist to complete

- Run Notebook 1 first if the synthetic CSV files are missing
- Successfully download one CelesTrak example and inspect its fields
- Install `sgp4` and run the propagation section
- Successfully download one Horizons benchmark table
- Save the standardized CSV files
- Decide which benchmark object should be kept for the paper
- Write down the differences between:
  - a synthetic state trajectory
  - a TLE
  - a propagated TLE time series
  - a Horizons state-vector table

## 15. Reflection question
Why is it important to convert synthetic data, propagated TLE data, and benchmark ephemerides into one common schema **before** training a PINN?


In [21]:

import json, numpy as np
_m = dict()
try:
    _m['celestrak_n'] = int(len(celestrak_standard))
    _m['celestrak_object'] = str(celestrak_standard['object_name'].iloc[0])
    _m['celestrak_min_alt_km'] = float(celestrak_standard['altitude_km'].min())
    _m['celestrak_max_alt_km'] = float(celestrak_standard['altitude_km'].max())
    _m['celestrak_mean_alt_km'] = float(celestrak_standard['altitude_km'].mean())
    _m['celestrak_median_step_sec'] = float(celestrak_standard['t_sec'].diff().median())
    _m['tle_epoch_utc'] = str(tle_epoch_ts)
except Exception as e:
    _m['celestrak_error'] = str(e)
try:
    row = celestrak_omm_df.iloc[0]
    for k in ['OBJECT_NAME','NORAD_CAT_ID','EPOCH','INCLINATION','ECCENTRICITY','MEAN_MOTION','BSTAR']:
        _m['omm_'+k] = str(row[k])
except Exception as e:
    _m['omm_error'] = str(e)
try:
    _m['horizons_n'] = int(len(horizons_standard))
    _m['horizons_object'] = str(horizons_standard['object_name'].iloc[0])
    _m['horizons_min_radius_km'] = float(horizons_standard['radius_km'].min())
    _m['horizons_max_radius_km'] = float(horizons_standard['radius_km'].max())
    _m['horizons_step_sec'] = float(horizons_standard['t_sec'].diff().median())
except Exception as e:
    _m['horizons_error'] = str(e)
try:
    _m['synthetic_standard_n'] = int(len(synthetic_standard)) if synthetic_standard is not None else 0
except Exception as e:
    _m['synthetic_error'] = str(e)
json.dump(_m, open('/tmp/pinn_run/results/nb2_metrics.json','w'), indent=2)
print('NB2_METRICS', json.dumps(_m, indent=2))


NB2_METRICS {
  "celestrak_n": 145,
  "celestrak_object": "ISS (ZARYA)",
  "celestrak_min_alt_km": 412.40407781499107,
  "celestrak_max_alt_km": 423.8810905648834,
  "celestrak_mean_alt_km": 418.4103150001601,
  "celestrak_median_step_sec": 600.0,
  "tle_epoch_utc": "2026-06-18 19:14:58.749487162+00:00",
  "omm_OBJECT_NAME": "ISS (ZARYA)",
  "omm_NORAD_CAT_ID": "25544",
  "omm_EPOCH": "2026-06-18 19:14:58.749504+00:00",
  "omm_INCLINATION": "51.6331",
  "omm_ECCENTRICITY": "0.00046669",
  "omm_MEAN_MOTION": "15.49302471",
  "omm_BSTAR": "0.00015438664",
  "horizons_n": 73,
  "horizons_object": "Moon",
  "horizons_min_radius_km": 360347.92394012853,
  "horizons_max_radius_km": 364083.39086056006,
  "horizons_step_sec": 3600.0,
  "synthetic_standard_n": 2500
}


In [22]:
print('FIG_FILES', sorted(__import__('os').listdir('figs')))

FIG_FILES ['nb1_fig01.png', 'nb1_fig02.png', 'nb1_fig03.png', 'nb1_fig04.png', 'nb2_fig01.png', 'nb2_fig02.png', 'nb2_fig03.png', 'nb2_fig04.png', 'nb2_fig05.png', 'nb2_fig06.png', 'nb2_fig07.png', 'nb3_fig01.png', 'nb3_fig02.png', 'nb3_fig03.png', 'nb3_fig04.png', 'nb3_fig05.png', 'nb3_fig06.png', 'nb3_fig07.png', 'nb3_fig08.png', 'nb3_fig09.png', 'nb3_fig10.png']
